# Regime Engine v2 — Calibration Index

One notebook per round, in chronological order. Each round records the table+plot+question+conclusion exchange.

## Rounds

* [Q1 + Q2 — macro scale fix and concept aggregation](regime_v2_calibration_q1q2.ipynb)
    * Q1: switched active growth FRED series from `none` → rolling z-score; clipped the threshold-divide normalization. Macro abs-p95 5.4 → 1.4.
    * Q2: replaced fast/slow buckets with concept aggregation (labor, consumption, production, broad_leading | realized_broad, persistence, market_expectations, wage_pressure). Per-series tanh compression.
* [Q3 — market-layer tanh, lower thresholds, label hysteresis](regime_v2_calibration_q3.ipynb)
    * Symmetric tanh on market layer; both layers in (-1, 1).
    * Axis thresholds ±0.20 + 5-day Up/Neutral/Down hysteresis.
    * AHETPI re-centered 4.0 → 3.0.
    * Regime turnover: 37.6/year → 9.3/year, median run 3 bdays → 18.
* [Q4 — market concept aggregation, beta-adjusted returns, S&P GSCI](regime_v2_calibration_q4.ipynb)
    * Concept aggregation on market: broad_equity_momentum, cyclical_rotation, risk_appetite, commodity_demand | commodity_inflation, inflation_pricing_proxy, gold_haven_inflation_proxy | volatility, credit_stress, flight_to_quality.
    * Beta-adjusted relative returns for QQQ/IWM vs SPY and the sector pairs.
    * S&P GSCI swapped in as the primary commodity-inflation member.
    * TLT/SPY dropped from growth (was 0.73 correlated with broad equity); kept in risk overlay as flight_to_quality.
* [Q5 — calibration workflow macro-config fix and baseline rebuild](regime_v2_calibration_q5.ipynb)
    * `run_regime_v2_calibration()` now loads and passes the macro-method config from `fred_series.yml` instead of falling back to `MacroRegimeConfig()` defaults.
    * Rebuilt the calibration baseline: 2022 and 2023-24 stop looking structurally inflation-hot simply because macro compression was missing in calibration.
    * Deferred threshold/weight retuning to the next round; Q5 is a baseline-correction pass, not a tuning pass.
* [Q6 — recovery responsiveness tuning](regime_v2_calibration_q6.ipynb)
    * Shifted the blend modestly toward `market_implied`: growth `0.35 / 0.65`, inflation `0.30 / 0.70` for macro / market.
    * Narrowed the axis deadband to growth `+/-0.15` and inflation `+/-0.12`.
    * 2017 moves to Goldilocks and 2020H2-2021 moves to Reflation while 2022 and 2023-24 remain neutral-majority.
* [Q7 — audit and measurement pass](regime_v2_calibration_q7.ipynb)
    * No tuning changes. Q7 reruns Q1-Q6 anchor summaries under the corrected Q5 workflow and Q6 baseline.
    * Adds `data_mode` semantics so latest market-only rows are explicit when macro data ends earlier than market data.
    * Adds phase counts, target-match share, first-match date, and response-lag metrics so majority labels do not hide timing.
    * Marks Q1-Q4 decisions as `still_valid`, `superseded`, or `needs_retest` against current evidence.
* [Q8 — macro release-recency responsiveness](regime_v2_calibration_q8.ipynb)
    * Macro panel now carries `_age_bdays:{series_id}` release-age columns.
    * Macro concept aggregation multiplies within-concept weights by recency weights and scales concept weights by availability.
    * Enabled mild recency weighting (`42` bday half-life, `0.65` floor) and added weekly initial claims (`ICSA`) inside the labor concept.
    * Preserves Q6 recovery readout while making stale monthly macro observations less dominant between release dates.

## Engine state

* Macro config: [`configs/regime_detection/fred_series.yml`](../../configs/regime_detection/fred_series.yml)
* Market config: [`configs/regime_detection/market_regime.yml`](../../configs/regime_detection/market_regime.yml)
* Engine config: [`configs/regime_detection/regime_engine.yml`](../../configs/regime_detection/regime_engine.yml)
* Output: `data/artifacts/regime_detection/regime_snapshots.json`
* Round snapshots: `regime_snapshots_q0_baseline.json` (pre-Q1) → `_q1.json` → `_q2.json` → `_q3.json` → `regime_snapshots.json` (current, post-Q6 recovery tuning plus Q7 audit fields)
